In [7]:
# ================== BLOCK 1 ==================
# Imports and initial setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
import warnings
import gc

warnings.filterwarnings('ignore')

# File path and chunk size
file_path = "archive/full_df.csv"
chunksize = 50_000

print("="*80)
print("COMPLETE FEATURE ANALYSIS - ENTIRE DATASET")
print("ALL ROWS USED FOR RANDOM FOREST AND CORRELATION")
print("="*80)


COMPLETE FEATURE ANALYSIS - ENTIRE DATASET
ALL ROWS USED FOR RANDOM FOREST AND CORRELATION


In [8]:
print("\n" + "="*80)
print("STEP 1: SCANNING ENTIRE DATASET")
print("="*80)

print("\n🔄 First pass: Collecting statistics from ALL rows...")
total_rows = 0
column_names = []
column_types = {}
value_counts = {}
running_min = {}
running_max = {}
running_sum = {}
running_count = {}
first_chunk = True

for i,chunk in enumerate(pd.read_csv(file_path, chunksize=chunksize)):
    if  i %20 ==0:
        print(f"   Chunk {i+1}... (rows: {total_rows:,})")
    if first_chunk:
        column_names = chunk.columns.tolist()
        for col in column_names:
            column_types[col] = str(chunk[col].dtype)
            value_counts[col] = set()
            running_min[col] = float('inf')
            running_max[col] = float('-inf')
            running_sum[col] = 0
            running_count[col] = 0
        first_chunk = False

    for col in column_names:
        if len(value_counts[col]) <= 1000:
            try:
                value_counts[col].update(chunk[col].dropna().unique()[:1000])
            except:
                pass

        if 'int' in str(chunk[col].dtype) or 'float' in str(chunk[col].dtype):
            try:
                valid_data = pd.to_numeric(chunk[col], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
                if len(valid_data) > 0:
                    running_min[col] = min(running_min[col], valid_data.min())
                    running_max[col] = max(running_max[col], valid_data.max())
                    running_sum[col] += valid_data.sum()
                    running_count[col] += len(valid_data)
            except:
                pass

    total_rows += len(chunk)
print(f"\n✅ Scanned ALL {total_rows:,} rows")

label_col='Label'
print(f"✅ Label column: {label_col}")


numeric_features = [col for col, dtype in column_types.items()
                   if ('int' in dtype or 'float' in dtype) and col != label_col]
non_numeric_features = [col for col, dtype in column_types.items()
                       if ('object' in dtype or 'string' in dtype) and col != label_col]
print(f"\n📊 Dataset:")
print(f"   Total rows: {total_rows:,}")
print(f"   Features: {len(column_names) - 1}")
print(f"   Numeric: {len(numeric_features)}")
print(f"   Non-numeric: {len(non_numeric_features)}")


#global medians for imputation
print("\n" + "="*80)
print("\n🔄 Calculating global medians for imputation...")
global_medians = {}
for col in numeric_features:
    if running_count[col] > 0:
        global_medians[col] = running_sum[col] / running_count[col]
    else:
        global_medians[col] = 0

print("\n🔄 Scanning all unique labels in dataset...")
all_labels = set()
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunksize)):
    if i % 50 == 0:
        print(f"   Scanning labels chunk {i+1}...")
    all_labels.update(chunk[label_col].unique())

all_labels = sorted(list(all_labels))
print(f"✅ Found {len(all_labels)} unique labels: {all_labels}")

# Fit label encoder
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)
print(f"✅ Label encoder fitted on all classes")



STEP 1: SCANNING ENTIRE DATASET

🔄 First pass: Collecting statistics from ALL rows...
   Chunk 1... (rows: 0)
   Chunk 21... (rows: 1,000,000)
   Chunk 41... (rows: 2,000,000)
   Chunk 61... (rows: 3,000,000)
   Chunk 81... (rows: 4,000,000)
   Chunk 101... (rows: 5,000,000)
   Chunk 121... (rows: 6,000,000)
   Chunk 141... (rows: 7,000,000)
   Chunk 161... (rows: 8,000,000)
   Chunk 181... (rows: 9,000,000)
   Chunk 201... (rows: 10,000,000)
   Chunk 221... (rows: 11,000,000)
   Chunk 241... (rows: 12,000,000)
   Chunk 261... (rows: 13,000,000)
   Chunk 281... (rows: 14,000,000)
   Chunk 301... (rows: 15,000,000)
   Chunk 321... (rows: 16,000,000)

✅ Scanned ALL 16,233,002 rows
✅ Label column: Label

📊 Dataset:
   Total rows: 16,233,002
   Features: 83
   Numeric: 82
   Non-numeric: 1


🔄 Calculating global medians for imputation...

🔄 Scanning all unique labels in dataset...
   Scanning labels chunk 1...
   Scanning labels chunk 51...
   Scanning labels chunk 101...
   Scanning labe

In [9]:
# ================== STEP 2: ZERO VARIANCE FEATURES ==================
print("\n" + "="*80)
print("STEP 2: ZERO VARIANCE FEATURES (ALL ROWS)")
print("="*80)

zero_variance_features = []

for col in column_names:
    if col == label_col:
        continue

    unique_count = len(value_counts[col])

    # حالة القيم الثابتة
    if unique_count <= 1:
        zero_variance_features.append({
            'Feature': col,
            'Unique_Values': unique_count,
            'Constant_Value': list(value_counts[col])[0] if unique_count == 1 else None
        })
    # حالة الأرقام الثابتة
    elif col in numeric_features and running_min[col] != float('inf') and running_min[col] == running_max[col]:
        zero_variance_features.append({
            'Feature': col,
            'Unique_Values': 1,
            'Constant_Value': running_min[col]
        })

# عرض وحفظ Zero Variance Features
if zero_variance_features:
    print(f"\n⚠️ Found {len(zero_variance_features)} zero variance features")
    zero_var_df = pd.DataFrame(zero_variance_features)
    print(zero_var_df)  # عرض الجدول على الشاشة
    zero_var_df.to_csv('zero_variance_features_full.csv', index=False)
else:
    print("✅ No zero variance features")
    zero_var_df = pd.DataFrame()

# ================== STEP 3: NON-NUMERIC FEATURES ==================
print("\n" + "="*80)
print("STEP 3: NON-NUMERIC FEATURES")
print("="*80)

non_numeric_keep = []
""" useless deja time ya3mil dataleackage will be dropped anyway  no non numerical feature"""

if non_numeric_features:
    print(f"\n⚠️ Found {len(non_numeric_features)} non-numeric features")

    for col in non_numeric_features:
        is_timestamp = 'time' in col.lower() or 'date' in col.lower()

        if is_timestamp:
            print(f"❌ Dropping timestamp column: {col}")
            continue

        non_numeric_keep.append(col)

    print(f"\n✅ Remaining non-numeric features: {non_numeric_keep}")

    non_numeric_info = []
    for col in non_numeric_keep:
        unique_count = len(value_counts.get(col, []))
        non_numeric_info.append({
            'Feature': col,
            'Type': column_types[col],
            'Unique_Values': unique_count,
            'Reason': "Non-numeric"
        })

    non_numeric_df = pd.DataFrame(non_numeric_info)
    print(non_numeric_df)
    non_numeric_df.to_csv('non_numeric_features_no_timestamp.csv', index=False)

else:
    print("✅ All features are numeric")
    non_numeric_df = pd.DataFrame()






STEP 2: ZERO VARIANCE FEATURES (ALL ROWS)

⚠️ Found 8 zero variance features
            Feature  Unique_Values  Constant_Value
0     Bwd PSH Flags              1               0
1     Bwd URG Flags              1               0
2    Fwd Byts/b Avg              1               0
3    Fwd Pkts/b Avg              1               0
4  Fwd Blk Rate Avg              1               0
5    Bwd Byts/b Avg              1               0
6    Bwd Pkts/b Avg              1               0
7  Bwd Blk Rate Avg              1               0

STEP 3: NON-NUMERIC FEATURES

⚠️ Found 1 non-numeric features
❌ Dropping timestamp column: Timestamp

✅ Remaining non-numeric features: []
Empty DataFrame
Columns: []
Index: []


will be removed + date time


In [12]:
# ================== BLOCK 4 ==================
print("\n" + "="*80)
print("STEP 4: TRAIN/TEST SPLIT AND RANDOM FOREST")
print("="*80)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# ================== ACCUMULATE DATA ==================
print("\n🔄 Loading data in chunks...")
all_X_chunks = []
all_y_chunks = []
total_rows_loaded = 0

for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunksize)):
    if i % 20 == 0:
        print(f"   Loading chunk {i+1}... ({total_rows_loaded:,} rows)")

    chunk_y = chunk[label_col]
    chunk_X = chunk[numeric_features].copy()

    # Clean data
    for col in chunk_X.columns:
        chunk_X[col] = pd.to_numeric(chunk_X[col], errors='coerce')
        chunk_X[col] = chunk_X[col].replace([np.inf, -np.inf], np.nan)
        chunk_X[col] = chunk_X[col].fillna(global_medians.get(col, 0))

    all_X_chunks.append(chunk_X)
    all_y_chunks.append(chunk_y)
    total_rows_loaded += len(chunk)

print(f"\n   Concatenating all data...")
X_full = pd.concat(all_X_chunks, ignore_index=True)
y_full = pd.concat(all_y_chunks, ignore_index=True)

del all_X_chunks, all_y_chunks
gc.collect()

print(f"✅ Loaded {len(X_full):,} rows × {len(numeric_features)} features")

# ================== ENCODE LABELS ==================
y_encoded = label_encoder.transform(y_full)

# ================== TRAIN/TEST SPLIT ==================
print(f"\n🔄 Splitting data (80% train, 20% test)...")
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # Keep class distribution
)

print(f"✅ Train set: {len(X_train):,} rows")
print(f"✅ Test set:  {len(X_test):,} rows")

# Free memory
del X_full, y_full, y_encoded
gc.collect()

# ================== TRAIN RANDOM FOREST ==================
print("\n" + "="*80)
print("TRAINING RANDOM FOREST")
print("="*80)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print(f"\n🌲 Training on {len(X_train):,} samples...")
rf.fit(X_train, y_train)

print(f"\n✅ Training complete!")

# ================== EVALUATE MODEL ==================
print("\n" + "="*80)
print("MODEL EVALUATION")
print("="*80)

print(f"\n🔄 Predicting on test set...")
y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n📊 Test Accuracy: {accuracy*100:.2f}%")

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# ================== FEATURE IMPORTANCE ==================
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

feature_importance = pd.DataFrame({
    'Feature': numeric_features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

feature_importance['Cumulative_Importance'] = feature_importance['Importance'].cumsum()
feature_importance['Rank'] = range(1, len(feature_importance) + 1)
feature_importance['Importance_Percent'] = (feature_importance['Importance'] * 100).round(2)

# Save
feature_importance.to_csv('feature_importance_with_test.csv', index=False)
print(f"✅ Feature importance saved to 'feature_importance_with_test1.csv'")

# Display top 30
print(f"\n📊 TOP 30 MOST IMPORTANT FEATURES:")
print("="*80)
top_30 = feature_importance.head(30)
for idx, row in top_30.iterrows():
    print(f"{row['Rank']:3d}. {row['Feature']:40s} {row['Importance_Percent']:6.2f}% (cumulative: {row['Cumulative_Importance']*100:6.2f}%)")

# Cumulative importance thresholds
print(f"\n📈 Features needed for importance thresholds:")
for threshold in [0.50, 0.80, 0.90, 0.95, 0.99]:
    n_features = (feature_importance['Cumulative_Importance'] <= threshold).sum() + 1
    print(f"   {threshold*100:.0f}% importance: {n_features} features")

print("\n" + "="*80)
print("✅ FEATURE ANALYSIS COMPLETE!")
print("="*80)


STEP 4: TRAIN/TEST SPLIT AND RANDOM FOREST

🔄 Loading data in chunks...
   Loading chunk 1... (0 rows)
   Loading chunk 21... (1,000,000 rows)
   Loading chunk 41... (2,000,000 rows)
   Loading chunk 61... (3,000,000 rows)
   Loading chunk 81... (4,000,000 rows)
   Loading chunk 101... (5,000,000 rows)
   Loading chunk 121... (6,000,000 rows)
   Loading chunk 141... (7,000,000 rows)
   Loading chunk 161... (8,000,000 rows)
   Loading chunk 181... (9,000,000 rows)
   Loading chunk 201... (10,000,000 rows)
   Loading chunk 221... (11,000,000 rows)
   Loading chunk 241... (12,000,000 rows)
   Loading chunk 261... (13,000,000 rows)
   Loading chunk 281... (14,000,000 rows)
   Loading chunk 301... (15,000,000 rows)
   Loading chunk 321... (16,000,000 rows)

   Concatenating all data...
✅ Loaded 16,233,002 rows × 82 features

🔄 Splitting data (80% train, 20% test)...
✅ Train set: 12,986,401 rows
✅ Test set:  3,246,601 rows

TRAINING RANDOM FOREST

🌲 Training on 12,986,401 samples...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.


building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100


KeyboardInterrupt: 